Regression Model

In [8]:

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Create sample data
# House size in square feet
X = np.array([
    [500], [700], [800], [1000], [1200],
    [1400], [1600], [1800], [2000], [2200],
    [2400], [2600], [2800], [3000], [3200],
    [3400], [3600], [3800], [4000], [4200]
], dtype=np.float32)

# Illustrative house prices in thousands of dollars
# Relationship: price = 50 + 0.2 * size
y = 50 + 0.2 * X[:, 0]
y = y.astype(np.float32)

# 2. Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Build the regression model
model = keras.Sequential([
    layers.Input(shape=(1,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1)  # Linear output for regression
])

# 4. Compile the model
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

# 5. Train the model
history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=4,
    verbose=0,
    validation_split=0.2
)

# 6. Evaluate on unseen test data
test_loss, test_mae = model.evaluate(
    X_test, y_test, verbose=0
)

print("Test MSE:", test_loss)
print("Test MAE:", test_mae)

# 7. Predict the price of a 2500 sq ft house
new_house = np.array([[2500]], dtype=np.float32)
prediction = model.predict(new_house, verbose=0)

print("Predicted price ($1000s):", prediction[0][0])
print("Predicted price ($):", prediction[0][0] * 1000)


Test MSE: 845.6558837890625
Test MAE: 25.071243286132812
Predicted price ($1000s): 542.1906
Predicted price ($): 542190.6


Classification Model

In [9]:

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Create sample data
# Each row represents study hours and attendance percentage
X = np.array([
    [1, 40], [2, 45], [1, 50], [3, 50],
    [2, 60], [4, 55], [3, 65], [4, 70],
    [5, 60], [5, 75], [6, 70], [6, 80],
    [7, 75], [7, 85], [8, 80], [8, 90],
    [9, 85], [9, 95], [10, 90], [10, 100],
    [2, 40], [3, 45], [4, 50], [5, 55],
    [6, 60], [7, 65], [8, 70], [9, 75],
    [1, 35], [2, 35], [3, 40], [4, 45],
    [5, 65], [6, 75], [7, 80], [8, 85],
    [9, 90], [10, 95], [3, 55], [6, 85]
], dtype=np.float32)

# 0 = Fail, 1 = Pass
y = np.array([
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 1, 0, 1, 1, 1, 1, 1, 1, 1,
    0, 0, 0, 0, 0, 0, 1, 1, 0, 0,
    0, 0, 0, 1, 1, 1, 1, 1, 0, 1
], dtype=np.float32)

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# 3. Normalize the features
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(X_train)

# 4. Build the classification model
model = keras.Sequential([
    layers.Input(shape=(2,)),
    normalizer,
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

# 5. Compile the model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# 6. Train
model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=4,
    verbose=0,
    validation_split=0.2
)

# 7. Predict probabilities
probabilities = model.predict(X_test, verbose=0).ravel()

# Convert probabilities to classes using threshold 0.5
y_pred = (probabilities >= 0.5).astype(int)

# 8. Evaluate
print("Test loss and accuracy:")
print(model.evaluate(X_test, y_test, verbose=0))

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# 9. Predict for a new student
# 7 study hours, 80% attendance
new_student = np.array([[7, 80]], dtype=np.float32)

probability = model.predict(new_student, verbose=0)[0][0]
predicted_class = int(probability >= 0.5)

print("\nPass probability:", probability)
print("Prediction:", "Pass" if predicted_class == 1 else "Fail")


Test loss and accuracy:
[0.07447850704193115, 1.0]

Accuracy: 1.0

Confusion Matrix:
[[6 0]
 [0 4]]

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00         6
         1.0       1.00      1.00      1.00         4

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10


Pass probability: 0.87746996
Prediction: Pass
